# Phase 2 — PINN Black-Scholes paramétrique

Ce notebook documente le livrable Phase 2 : un modèle paramétrique capable de pricer une option européenne call Black-Scholes sur un espace de paramètres variable.

## Objectifs

- Généraliser la Phase 1 à un domaine paramétrique `(S, K, sigma, r, T, tau)`.
- Utiliser la formulation log-moneyness `x = log(S / K)`.
- Comparer le modèle à la formule analytique Black-Scholes.
- Ajouter un benchmark de vitesse complet incluant Monte Carlo.


## Formulation

Variables réseau :

```text
x = log(S / K)
tau / T
sigma
r
T
K_norm optionnel, ignoré dans les meilleures expériences
```

Sortie normalisée :

```text
u = V / K
V = K * u
```

PDE log-moneyness :

```text
u_tau - 0.5 sigma^2 u_xx - (r - 0.5 sigma^2) u_x + r u = 0
```


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'neuroprice').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

PROJECT_ROOT


## Commandes de reproduction

Meilleur entraînement PINN/surrogate paramétrique avec Fourier features :

```bash
python -m scripts.train_pinn_bs_parametric --output-transform direct --fourier-features 4 --pretrain-epochs 10000 --pretrain-lr 0.001 --pretrain-samples 32768 --pretrain-relative-weight 1.0 --relative-floor 0.01 --epochs 12000 --hidden-dim 256 --hidden-layers 5 --n-interior 12000 --n-terminal 6000 --n-boundary 6000 --n-supervised 12000 --supervised-weight 0.05 --supervised-relative-weight 1.0 --lbfgs-steps 500 --pde-weight 1 --terminal-weight 10 --lower-boundary-weight 1 --upper-boundary-weight 1 --out-dir artifacts/phase2_bs_pinn_parametric_fourier
```

Validation avec benchmark Monte Carlo :

```bash
python -m scripts.validate_pinn_bs_parametric --checkpoint artifacts/phase2_bs_pinn_parametric_fourier/bs_pinn_parametric.pt --out artifacts/phase2_bs_pinn_parametric_fourier/benchmark_with_mc.json --n-points 10000 --relative-floor 1.0 --min-reference-price 1.0 --mc-paths 10000 --mc-chunk-size 1000
```


## Chargement du benchmark

Cette cellule charge `benchmark_with_mc.json` si l'artefact existe. Sinon, elle rappelle la commande à exécuter.


In [ ]:
benchmark_path = PROJECT_ROOT / 'artifacts' / 'phase2_bs_pinn_parametric_fourier' / 'benchmark_with_mc.json'
fallback_path = PROJECT_ROOT / 'artifacts' / 'phase2_bs_pinn_parametric_fourier' / 'benchmark.json'
path = benchmark_path if benchmark_path.exists() else fallback_path

if path.exists():
    benchmark = json.loads(path.read_text(encoding='utf-8'))
    selected = {
        key: benchmark.get(key)
        for key in [
            'n_points',
            'relative_l2',
            'p95_point_relative_error',
            'tradable_p95_point_relative_error',
            'pct_points_under_0_5pct_error',
            'tradable_pct_points_under_0_5pct_error',
            'pinn_points_per_second',
            'analytic_points_per_second',
            'monte_carlo_points_per_second',
            'pinn_vs_monte_carlo_speedup',
            'monte_carlo_relative_l2',
        ]
    }
    pd.DataFrame([selected]).T.rename(columns={0: 'value'})
else:
    print('Benchmark Phase 2 absent. Exécuter la commande de validation ci-dessus.')


## Critères suivis

| Critère | Cible | Métrique |
| --- | ---: | --- |
| Précision prix | 95% des points sous 0.5% | `pct_points_under_0_5pct_error` |
| Vitesse absolue | PINN >= 50k prix/s | `pinn_points_per_second` |
| Accélération vs MC | PINN >= 100x MC | `pinn_vs_monte_carlo_speedup` |
| Comparaison analytique | PINN proche de Black-Scholes | `relative_l2` |


In [ ]:
if path.exists():
    checks = {
        'pinn_speed_ge_50k': benchmark.get('pinn_points_per_second', 0.0) >= 50_000,
        'speedup_vs_mc_ge_100x': benchmark.get('pinn_vs_monte_carlo_speedup', 0.0) >= 100,
        'has_mc_benchmark': benchmark.get('monte_carlo_points_per_second') is not None,
    }
    checks
else:
    {}


## Conclusion Phase 2

La Phase 2 fournit un pipeline paramétrique complet :

- modèle `ParametricBlackScholesPINN`,
- domaine multi-paramètres,
- pré-entraînement supervisé analytique,
- fine-tuning PDE optionnel,
- Fourier features,
- validation analytique,
- benchmark vitesse PINN vs analytique vs Monte Carlo.

Ce notebook est le support de synthèse Phase 2 avant le passage aux options exotiques de Phase 3.
